# NFL Roster Construction & Cap Efficiency
### 01: Data Collection

Pulls rosters and contracts, merges them using standardized position GROUPS,
and aggregates to team + position group + season.

**Note:** unlike the Player Value Engine, this notebook does NOT pull
`import_seasonal_data()` (offensive/fantasy stats). That dataset only
contains passing/rushing/receiving production, which excludes most CBs,
LBs, safeties, and offensive linemen almost entirely (they don't generate
those stat types). Using it as a base would silently drop most of the
roster before the contract merge even happens. This project only needs
roster membership (position, team, season) and contract value, so
`import_seasonal_rosters()` is used as the base instead.

In [28]:
import nfl_data_py as nfl
import pandas as pd
import os

YEARS = list(range(2013, 2025))  # 2013-2024, confirmed working (2025 not yet published)

print("Pulling roster data...")
rosters = nfl.import_seasonal_rosters(years=YEARS)
print(f"Rosters shape: {rosters.shape}")

Pulling roster data...
Rosters shape: (34338, 37)


In [29]:
# Keep only the columns needed from rosters
roster_clean = rosters[['player_id', 'season', 'player_name', 'position',
                          'age', 'years_exp', 'draft_number', 'entry_year',
                          'team', 'weight', 'height']].copy()

roster_clean = roster_clean.drop_duplicates(subset=['player_id', 'season'])

print(f"Roster (cleaned) shape: {roster_clean.shape}")
print(f"Season range: {roster_clean['season'].min()} - {roster_clean['season'].max()}")
print(f"\nRaw positions in roster data:\n{sorted(roster_clean['position'].dropna().unique())}")

Roster (cleaned) shape: (34009, 11)
Season range: 2013 - 2024

Raw positions in roster data:
['C', 'CB', 'DB', 'DE', 'DL', 'DT', 'FB', 'FS', 'G', 'ILB', 'K', 'LB', 'LS', 'MLB', 'NT', 'OL', 'OLB', 'P', 'PR', 'QB', 'RB', 'S', 'SS', 'T', 'TE', 'WR']


In [30]:
print("Pulling contract data...")
contracts = nfl.import_contracts()
print(f"Contracts shape: {contracts.shape}")

contracts_clean = contracts[['player', 'position', 'team', 'year_signed',
                               'years', 'value', 'apy', 'guaranteed',
                               'draft_overall']].copy()

contracts_clean = contracts_clean.rename(columns={'player': 'player_name', 'apy': 'aav'})

# Drop rows with missing/placeholder year_signed (e.g. 0) before merging
contracts_clean = contracts_clean[contracts_clean['year_signed'] > 2000]
print(f"Contracts after cleaning: {contracts_clean.shape}")
print(f"\nRaw positions in contract data:\n{sorted(contracts_clean['position'].dropna().unique())}")

Pulling contract data...
Contracts shape: (51982, 26)
Contracts after cleaning: (50666, 9)

Raw positions in contract data:
['C', 'CB', 'ED', 'FB', 'IDL', 'K', 'LB', 'LG', 'LS', 'LT', 'P', 'QB', 'RB', 'RG', 'RT', 'S', 'TE', 'WR']


### Position group mapping

Rolling individual positions up to standard front-office position groups.

**Why this matters here:** the roster data and contract data come from
different underlying sources and don't necessarily use identical position
labels for the same role (e.g. one source may list offensive tackles as
`OT`, the other as `T`, or edge rushers as `DE` vs `ED`). Merging on the
raw `position` string requires an exact match, so any inconsistency
silently drops those players from the merge entirely — no error, just
missing rows. Mapping BOTH sources to the same position-group buckets
before merging avoids this.

In [31]:
position_group_map = {
    'QB': 'QB',
    'RB': 'RB', 'FB': 'RB', 'HB': 'RB',
    'WR': 'WR',
    'TE': 'TE',
    'T': 'OL', 'G': 'OL', 'C': 'OL', 'OL': 'OL', 'OT': 'OL', 'OG': 'OL', 'LT': 'OL', 'RT': 'OL', 'LG': 'OL', 'RG': 'OL',
    'DE': 'DL', 'DT': 'DL', 'NT': 'DL', 'DL': 'DL', 'EDGE': 'DL', 'ED': 'DL', 'IDL': 'DL',
    'LB': 'LB', 'OLB': 'LB', 'ILB': 'LB', 'MLB': 'LB',
    # Cornerbacks and safeties are combined into one group. The roster data
    # uses a generic 'DB' label for a large share of defensive backs that
    # doesn't reliably distinguish cornerback from safety, so splitting
    # them would mean guessing. Combining is the more honest approach.
    'CB': 'CB_S', 'DB': 'CB_S', 'S': 'CB_S', 'FS': 'CB_S', 'SS': 'CB_S', 'SAF': 'CB_S',
    'K': 'ST', 'P': 'ST', 'LS': 'ST', 'PR': 'ST', 'KR': 'ST',
}

roster_clean['position_group'] = roster_clean['position'].map(position_group_map)
contracts_clean['position_group'] = contracts_clean['position'].map(position_group_map)

print("Unmapped positions in roster data:", sorted(roster_clean[roster_clean['position_group'].isna()]['position'].dropna().unique()))
print("Unmapped positions in contract data:", sorted(contracts_clean[contracts_clean['position_group'].isna()]['position'].dropna().unique()))

Unmapped positions in roster data: []
Unmapped positions in contract data: []


If either list above isn't empty, add the missing raw position label(s)
to `position_group_map` and re-run this cell before continuing — an
unmapped position will be dropped from the merge below and would silently
exclude those players.

In [32]:
# Drop rows that couldn't be mapped to a position group on either side
roster_mapped = roster_clean.dropna(subset=['position_group'])
contracts_mapped = contracts_clean.dropna(subset=['position_group'])

print(f"Roster rows after dropping unmapped positions: {roster_mapped.shape[0]} (was {roster_clean.shape[0]})")
print(f"Contract rows after dropping unmapped positions: {contracts_mapped.shape[0]} (was {contracts_clean.shape[0]})")

print("\nRoster rows by position_group:")
print(roster_mapped['position_group'].value_counts())

print("\nContract rows by position_group:")
print(contracts_mapped['position_group'].value_counts())

Roster rows after dropping unmapped positions: 33993 (was 34009)
Contract rows after dropping unmapped positions: 50666 (was 50666)

Roster rows by position_group:
CB_S    6403
OL      5716
DL      5010
LB      4519
WR      4303
RB      2727
TE      2282
ST      1618
QB      1415
Name: position_group, dtype: int64

Contract rows by position_group:
CB_S    9780
DL      8926
OL      8296
WR      7354
LB      4713
RB      4123
TE      3409
QB      2092
ST      1973
Name: position_group, dtype: int64


### Leakage-safe merge

Merge on `player_name` + `position_group`, keeping only contracts signed
on or before the season they're matched to, and keeping the most recent
qualifying contract per player per season.

In [33]:
master_df = pd.merge(roster_mapped, contracts_mapped, on=['player_name', 'position_group'], how='inner')
master_df = master_df[master_df['year_signed'] <= master_df['season']]
master_df = master_df.sort_values('year_signed', ascending=False)
master_df = master_df.drop_duplicates(subset=['player_name', 'season'], keep='first')

print(f"Master dataset shape: {master_df.shape}")
print(f"Season range: {master_df['season'].min()} - {master_df['season'].max()}")
print(f"\nRows by position_group after merge:")
print(master_df['position_group'].value_counts())

Master dataset shape: (29270, 20)
Season range: 2013 - 2024

Rows by position_group after merge:
CB_S    5724
OL      5183
DL      4503
WR      3854
LB      2979
RB      2344
TE      1995
ST      1388
QB      1300
Name: position_group, dtype: int64


In [34]:
# Confirm no player appears more than once per season after the merge
duplicates = master_df.groupby(['player_name', 'season']).size().reset_index(name='count')
print(f"Max times a player appears in one season: {duplicates['count'].max()}")

master_df[['player_name', 'position_group', 'season', 'team_x', 'age', 'aav', 'guaranteed']].head(10)

Max times a player appears in one season: 1


,player_name,position_group,season,team_x,age,aav,guaranteed
205093,Camron Peterson,DL,2024,NO,24.0,0.943333,0.0
150641,Curtis Bolton,LB,2024,TEN,28.0,0.403300,0.0
193426,Sam Roberts,DL,2024,CAR,26.0,0.225000,0.0
170612,Trishton Jackson,WR,2024,MIN,26.0,0.985000,0.0
199633,Owen Wright,RB,2024,BAL,25.0,0.795000,0.0
101275,Joey Bosa,DL,2024,LAC,29.0,20.180000,15.0
193405,Cade Mays,OL,2024,CAR,25.0,0.225000,0.0
199647,Tykeem Doss,OL,2024,PIT,24.0,0.795000,0.0
185219,Royce Newman,OL,2024,TB,27.0,2.250000,0.0
185244,Caden Sterns,CB_S,2024,PHI,24.0,1.055000,0.0


### Team-level cap allocation by position group and season

In [35]:
team_cap_allocation = (
    master_df.groupby(['team_x', 'season', 'position_group'])['aav']
    .sum()
    .reset_index()
    .rename(columns={'team_x': 'team', 'aav': 'total_cap_allocated'})
)

print(f"Team cap allocation shape: {team_cap_allocation.shape}")

# Sanity check: every position group should have a plausible nonzero total.
# A position group that's near-zero or wildly out of proportion to the
# others indicates a mapping or merge problem upstream.
totals_check = team_cap_allocation.groupby('position_group')['total_cap_allocated'].sum()
print("\nTotal cap allocated by position group (should all be reasonably large, no group wildly out of line):")
print(totals_check.sort_values())

team_cap_allocation.head(10)

Team cap allocation shape: (3436, 4)

Total cap allocated by position group (should all be reasonably large, no group wildly out of line):
position_group
ST       1990.753370
RB       3698.267105
TE       3775.806476
LB       5596.377639
QB       8078.947990
WR       8662.535083
CB_S    12031.294511
DL      12328.566757
OL      13398.192759
Name: total_cap_allocated, dtype: float64


,team,season,position_group,total_cap_allocated
0,ARI,2016,CB_S,47.164287
1,ARI,2016,DL,25.541979
2,ARI,2016,LB,6.883084
3,ARI,2016,OL,27.289530
4,ARI,2016,QB,25.315000
5,ARI,2016,RB,3.301091
6,ARI,2016,ST,4.038000
7,ARI,2016,TE,6.095825
8,ARI,2016,WR,15.493175
9,ARI,2017,CB_S,41.372422


### Team performance data

Pulling game results to compute wins and point differential per team per
season — this becomes the outcome variable for the efficiency analysis.

In [36]:
print("Pulling schedule/results data...")
schedules = nfl.import_schedules(years=YEARS)
print(f"Schedules shape: {schedules.shape}")
schedules[['season', 'week', 'home_team', 'away_team', 'home_score', 'away_score']].head()

Pulling schedule/results data...
Schedules shape: (3277, 46)


,season,week,home_team,away_team,home_score,away_score
3714,2013,1,DEN,BAL,49.0,27.0
3715,2013,1,BUF,NE,21.0,23.0
3716,2013,1,CAR,SEA,7.0,12.0
3717,2013,1,CHI,CIN,24.0,21.0
3718,2013,1,CLE,MIA,10.0,23.0


In [37]:
# Build a team-season win/point-differential table from schedule results
completed = schedules.dropna(subset=['home_score', 'away_score']).copy()

home = completed[['season', 'home_team', 'home_score', 'away_score']].rename(
    columns={'home_team': 'team', 'home_score': 'points_for', 'away_score': 'points_against'})
away = completed[['season', 'away_team', 'away_score', 'home_score']].rename(
    columns={'away_team': 'team', 'away_score': 'points_for', 'home_score': 'points_against'})

team_games = pd.concat([home, away], ignore_index=True)
team_games['win'] = (team_games['points_for'] > team_games['points_against']).astype(int)

team_performance = team_games.groupby(['team', 'season']).agg(
    wins=('win', 'sum'),
    games=('win', 'count'),
    point_diff=('points_for', lambda x: x.sum() - team_games.loc[x.index, 'points_against'].sum())
).reset_index()

print(f"Team performance shape: {team_performance.shape}")
team_performance.head(10)

Team performance shape: (384, 5)


,team,season,wins,games,point_diff
0,ARI,2013,10,16,55.0
1,ARI,2014,11,17,0.0
2,ARI,2015,14,18,148.0
3,ARI,2016,7,16,56.0
4,ARI,2017,8,16,-66.0
5,ARI,2018,3,16,-200.0
6,ARI,2019,5,16,-81.0
7,ARI,2020,8,16,43.0
8,ARI,2021,11,18,60.0
9,ARI,2022,4,17,-109.0


### Standardize team abbreviations

Several franchises changed team codes during this window (relocations):
OAK -> LV (Raiders, 2020), SD -> LAC (Chargers, 2017), STL -> LA (Rams, 2016).
Standardizing both tables to each team's current code prevents these from
being treated as different teams later when merging cap allocation with
performance data.

In [38]:
team_code_fixes = {
    'OAK': 'LV',
    'SD': 'LAC',
    'STL': 'LA',
}

team_cap_allocation['team'] = team_cap_allocation['team'].replace(team_code_fixes)
team_performance['team'] = team_performance['team'].replace(team_code_fixes)

print("Unique teams in cap allocation:", team_cap_allocation['team'].nunique())
print("Unique teams in performance:", team_performance['team'].nunique())

Unique teams in cap allocation: 37
Unique teams in performance: 32


In [39]:
# Save intermediate outputs to data folder
os.makedirs('../data', exist_ok=True)

team_cap_allocation.to_csv('../data/team_cap_allocation.csv', index=False)
team_performance.to_csv('../data/team_performance.csv', index=False)

print("Saved team_cap_allocation.csv and team_performance.csv")

Saved team_cap_allocation.csv and team_performance.csv
